In [1]:
!pip install -q pinecone sentence-transformers groq ragas datasets langchain-community langchain fastapi uvicorn nest-asyncio pyngrok 


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.7/742.7 kB 13.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 64.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.7/502.7 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.9/280.9 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.4/177.4 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install -q langchain-groq faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 74.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.9 MB/s eta 0:00:00


In [3]:
!pip install -q chromadb


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 75.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 87.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. T

In [4]:
import os
import json
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
from pinecone import Pinecone, ServerlessSpec
import faiss
from groq import Groq
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from datasets import Dataset

# --- API KEYS (use Kaggle Secrets) ---
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

PINECONE_API_KEY = secrets.get_secret("PINECONE_API_KEY")
GROQ_API_KEY = secrets.get_secret("GROQ_API_KEY")

print("Keys loaded ✅")


/usr/local/lib/python3.12/dist-packages/wrapt/importer.py:223: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  self.__wrapped__.exec_module(module)


Keys loaded ✅


/tmp/ipykernel_55/2627303500.py:10: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
/tmp/ipykernel_55/2627303500.py:10: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
/tmp/ipykernel_55/2627303500.py:10: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import faithfulness, answer_relevancy, context_preci

## load dataset 

In [5]:
from datasets import load_dataset

dataset = load_dataset("lavita/MedQuAD", split="train")

print(f"Total samples: {len(dataset)}")
print(f"Columns: {dataset.column_names}")

print("\nExample:")
first = dataset[0]
print("Question:", first["question"])
print("\nAnswer (truncated):", first["answer"][:500])


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-e36383d177026d(…):   0%|          | 0.00/10.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/47441 [00:00<?, ? examples/s]

Total samples: 47441
Columns: ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer']

Example:
Question: What is (are) keratoderma with woolly hair ?

Answer (truncated): Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that ca


In [6]:
docs = []
for i, item in enumerate(dataset):
    question = item["question"]
    answer = item["answer"]
    label = item.get("question_type", "unknown")

    docs.append({
        "id": str(i),
        "question": question,
        "answer": answer,
        "label": label,
    })

print(f"Prepared {len(docs)} documents ✅")
print("\nSample doc:")
print(docs[0])


Prepared 47441 documents ✅

Sample doc:
{'id': '0', 'question': 'What is (are) keratoderma with woolly hair ?', 'answer': 'Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the soles of the feet to become thick, scaly, and calloused.  Cardiomyopathy, which is a disease of the heart muscle, is a life-threatening health problem that can develop in people with keratoderma with woolly hair. Unlike the other features of this condition, signs and symptoms of cardiomyopathy may not appear until adolescence or la

## generate emb

In [7]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")  # 384-dim embeddings[web:52]

texts = [
    f"Question: {d['question']}\nAnswer: {d['answer']}"
    for d in docs
]

print("Number of texts:", len(texts))
print("Example text:\n", texts[0][:500])


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Number of texts: 47441
Example text:
 Question: What is (are) keratoderma with woolly hair ?
Answer: Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individ


In [8]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)

embeddings = embedder.encode(
    texts,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/742 [00:00<?, ?it/s]

## Store in Pinecone

In [11]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

INDEX_NAME = "medical-rag"
DIMENSION = embeddings.shape[1]

existing_indexes = [i.name for i in pc.list_indexes()]
if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print(f"Index '{INDEX_NAME}' created ✅")

index = pc.Index(INDEX_NAME)
print("Pinecone index ready ✅")


Pinecone index ready ✅


In [12]:
embeddings = np.nan_to_num(embeddings)

batch_size = 100
for i in range(0, len(docs), batch_size):
    batch_docs = docs[i:i+batch_size]
    batch_embs = embeddings[i:i+batch_size]

    vectors = []
    for doc, emb in zip(batch_docs, batch_embs):
        meta = {
            "question": str(doc["question"])[:200],
            "answer": str(doc["answer"])[:200],
            "label": str(doc["label"]),
        }
        v = {
            "id": str(doc["id"]),
            "values": emb.tolist(),
            "metadata": meta,
        }
        vectors.append(v)

    if not vectors:
        continue

    index.upsert(vectors=vectors)

print(f"Upserted {len(docs)} vectors to Pinecone ✅")
print(index.describe_index_stats())


Upserted 47441 vectors to Pinecone ✅
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '189',
                                    'content-type': 'application/json',
                                    'date': 'Mon, 09 Mar 2026 13:41:08 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '40',
                                    'x-pinecone-request-latency-ms': '39',
                                    'x-pinecone-response-duration-ms': '41'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 48441}},
 'storageFullness': 0.0,
 'total_vector_count': 48441,
 'vector_type': 'dense'}


##  Build FAISS Index (local, for comparison)

In [13]:
def retrieve_faiss(query: str, top_k: int = 10):
    q_emb = embedder.encode(query, normalize_embeddings=True).astype(np.float32).reshape(1, -1)
    scores, indices = faiss_index.search(q_emb, top_k)
    out = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        doc = faiss_docs[idx]
        out.append({
            "text":     doc.get("answer", ""),
            "question": doc.get("question", ""),
            "score":    float(score),
        })
    return out


In [14]:
import faiss
import numpy as np

DIMENSION = embeddings.shape[1]  # 384

# Use distinct names — 'index' is already taken by Pinecone
faiss_index = faiss.IndexFlatIP(DIMENSION)
faiss_index.add(embeddings.astype(np.float32))
faiss_docs  = docs.copy()

print(f"FAISS index built with {faiss_index.ntotal} vectors")


FAISS index built with 47441 vectors


## chromadb

In [55]:
import chromadb

# ── 1. Client & collection ────────────────────────────────────────────────────
try:
    chroma_client.delete_collection("medical-rag")
except:
    chroma_client = chromadb.EphemeralClient()

chroma_collection = chroma_client.create_collection(
    name="medical-rag",
    metadata={"hnsw:space": "cosine"}
)

# ── 2. Build ID lookup + Upsert ───────────────────────────────────────────────
docs_by_id = {str(d["id"]): d for d in docs}   # ← fast full-doc lookup

BATCH_SIZE = 500
for i in range(0, len(docs), BATCH_SIZE):
    batch_docs = docs[i:i + BATCH_SIZE]
    batch_embs = embeddings[i:i + BATCH_SIZE]
    chroma_collection.add(
        ids        = [str(d["id"]) for d in batch_docs],
        embeddings = batch_embs.tolist(),
        documents  = [(d.get("answer") or "")[:1000] for d in batch_docs],
        metadatas  = [
            {
                "question": (d.get("question") or "")[:300],
                "answer":   (d.get("answer")   or "")[:200],
                "label":    (d.get("label")    or "unknown"),
                "doc_id":   str(d["id"]),        # ← stored for lookup
            }
            for d in batch_docs
        ],
    )

print(f"ChromaDB ready: {chroma_collection.count()} vectors ✓")

# ── 3. Retrieve function ──────────────────────────────────────────────────────
def retrieve_chroma(query: str, top_k: int = 10):
    q_emb = embedder.encode(query, normalize_embeddings=True).tolist()
    results = chroma_collection.query(
        query_embeddings=[q_emb],
        n_results=top_k,
        include=["metadatas", "distances", "documents"]   # ← no "ids"
    )
    out = []
    seen_texts = set()                                     # ← dedup
    for meta, dist, doc_text in zip(
        results["metadatas"][0],
        results["distances"][0],
        results["documents"][0]
    ):
        similarity = max(0.0, 1.0 - dist)
        if similarity < 0.3:
            continue

        # Full answer via doc_id stored in metadata
        doc_id    = meta.get("doc_id", "")
        full_doc  = docs_by_id.get(doc_id, {})
        full_text = (full_doc.get("answer") or doc_text or "")[:2000]

        if full_text in seen_texts:                        # skip duplicates
            continue
        seen_texts.add(full_text)

        out.append({
            "text":     full_text,
            "question": (full_doc.get("question") or meta.get("question") or ""),
            "score":    similarity,
        })
    return out

# ── 4. Sanity check ───────────────────────────────────────────────────────────
_test = retrieve_chroma("What are the effects of metformin on type 2 diabetes?", top_k=5)
print(f"\nRetrieval test — {len(_test)} unique results")
for i, r in enumerate(_test):
    print(f"  [{i+1}] score={r['score']:.3f} | {r['text'][:120]}")


ChromaDB ready: 47441 vectors ✓

Retrieval test — 5 unique results
  [1] score=0.735 | Yes. The results of the Diabetes Prevention Program (DPP) proved that weight loss through moderate diet changes and phys
  [2] score=0.721 | A major research study, the Diabetes Prevention Program (DPP), proved that people with prediabetes were able to sharply 
  [3] score=0.719 | You can do a lot to reduce your risk of getting type 2 diabetes. Being more physically active, reducing fat and calorie 
  [4] score=0.713 | Diabetes is a complex group of diseases with a variety of causes. People with diabetes have high blood glucose, also cal
  [5] score=0.707 | Type 2 diabetesthe most common form of diabetesis caused by a combination of factors, including insulin resistance, a co


In [ ]:
# def retrieve_chroma(query: str, top_k: int = 10):
#     q_emb = embedder.encode(query, normalize_embeddings=True).tolist()
#     results = chroma_collection.query(
#         query_embeddings=[q_emb],
#         n_results=top_k,
#         include=["metadatas", "distances"]
#     )
#     out = []
#     for meta, dist in zip(results["metadatas"][0], results["distances"][0]):
#         out.append({
#             "text":     meta.get("answer", ""),
#             "question": meta.get("question", ""),
#             "score":    1 - dist,   # ChromaDB returns L2/cosine distance → convert to similarity
#         })
#     return out


## Retrieval Functions

In [64]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# ── Fast ID lookup ────────────────────────────────────────────────────────────
docs_by_id = {str(d["id"]): d for d in docs}

# ── Pinecone ──────────────────────────────────────────────────────────────────
def retrieve_pinecone(query: str, top_k: int = 10):
    q_emb = embedder.encode(query, normalize_embeddings=True).tolist()
    results = index.query(vector=q_emb, top_k=top_k, include_metadata=True)
    out = []
    for m in results.matches:
        doc = docs_by_id.get(m.id, {})
        out.append({
            "text":     (doc.get("answer")   or m.metadata.get("answer", ""))[:2000],
            "question": (doc.get("question") or m.metadata.get("question", "")),
            "score":    m.score,
        })
    return out

# ── FAISS ─────────────────────────────────────────────────────────────────────
def retrieve_faiss(query: str, top_k: int = 10):
    q_emb = (
        embedder.encode(query, normalize_embeddings=True)
        .astype(np.float32)
        .reshape(1, -1)
    )
    scores, indices = faiss_index.search(q_emb, top_k)
    out = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue
        doc = faiss_docs[idx]
        out.append({
            "text":     (doc.get("answer")   or "")[:2000],
            "question": (doc.get("question") or ""),
            "score":    float(score),
        })
    return out

# ── ChromaDB ──────────────────────────────────────────────────────────────────
def retrieve_chroma(query: str, top_k: int = 10):
    q_emb = embedder.encode(query, normalize_embeddings=True).tolist()
    results = chroma_collection.query(
        query_embeddings=[q_emb],
        n_results=top_k,
        include=["metadatas", "distances", "documents"]   # ← NO "ids"
    )
    out = []
    seen_texts = set()
    for meta, dist, doc_text in zip(
        results["metadatas"][0],
        results["distances"][0],
        results["documents"][0]
    ):
        similarity = max(0.0, 1.0 - dist)
        if similarity < 0.3:
            continue
        doc_id    = meta.get("doc_id", "")
        full_doc  = docs_by_id.get(doc_id, {})
        full_text = (full_doc.get("answer") or doc_text or "")[:2000]
        if full_text in seen_texts:
            continue
        seen_texts.add(full_text)
        out.append({
            "text":     full_text,
            "question": (full_doc.get("question") or meta.get("question") or ""),
            "score":    similarity,
        })
    return out

# ── Re-ranker ─────────────────────────────────────────────────────────────────
def rerank(query: str, results: list, top_k: int = 5):
    if not results:
        return []
    pairs  = [(query, r["text"]) for r in results]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, results), key=lambda x: x[0], reverse=True)
    return [r for _, r in ranked[:top_k]]

print("Retrieval functions ready ✓")


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retrieval functions ready ✓


##  Generation with Groq

In [57]:
# groq_client = Groq(api_key=GROQ_API_KEY)

# def generate_answer(query: str, context_docs: list, model: str = "llama-3.3-70b-versatile"):
#     """Generate answer from retrieved context using Groq"""
#     context = "\n\n".join([
#         f"[Doc {i+1}] (relevance: {d['score']:.3f}):\n{d['text']}"
#         for i, d in enumerate(context_docs)
#     ])

#     system_prompt = """You are an expert medical research assistant trained on peer-reviewed literature.

# Your job is to answer clinical and biomedical questions accurately and concisely based ONLY on the provided context documents.

# Rules:
# - Answer directly and confidently when the context supports it
# - Cite which document(s) support your answer, e.g. "According to Doc 1..."
# - If multiple docs are relevant, synthesize them into a single coherent answer
# - If the context is insufficient, say: "The provided context does not contain enough information to answer this question fully." then state what IS known
# - Never fabricate facts or introduce outside knowledge
# - Keep answers focused: 3-5 sentences maximum
# - Use precise medical language appropriate for a clinical audience"""

#     user_prompt = f"""CONTEXT DOCUMENTS:
# {context}

# QUESTION: {query}

# Provide a concise, evidence-based answer citing the relevant documents."""

#     response = groq_client.chat.completions.create(
#         model=model,
#         messages=[
#             {"role": "system", "content": system_prompt},
#             {"role": "user", "content": user_prompt}
#         ],
#         temperature=0.1,
#         max_tokens=512
#     )
#     return response.choices[0].message.content

# print("Generation function ready ✅")


In [58]:
import re
groq_client = Groq(api_key=GROQ_API_KEY)

# Best available model on Groq for medical reasoning
GENERATION_MODEL = "openai/gpt-oss-120b"



def generate_answer(query: str, context_docs: list, model: str = GENERATION_MODEL):

    # Build rich context with full text + source question
    context = "\n\n".join([
        f"[Doc {i+1}] (similarity: {d['score']:.3f})\n"
        f"Source Q: {d['question']}\n"
        f"Content: {d['text']}"
        for i, d in enumerate(context_docs)
    ])

    system_prompt = """You are a senior clinical physician and biomedical researcher with 20 years of experience. You answer medical questions with authority, precision, and depth.

CORE INSTRUCTIONS:
- Write with CONFIDENCE. State facts directly. Do not hedge unnecessarily.
- Use the provided documents as your primary evidence base.
- Structure every answer with these sections:
  **Summary** — 1-2 sentence direct answer
  **Evidence** — What the documents specifically state, with citations (Doc 1, Doc 2...)
  **Clinical Context** — Mechanism, implications, or related considerations
- Cite documents inline: "According to Doc 1...", "Doc 2 confirms that..."
- If a document is only partially relevant, extract whatever IS useful from it
- NEVER say "the context does not contain enough information" unless ALL documents are completely off-topic
- Do NOT fabricate — if something is unknown, say "the provided documents do not specify X, but establish that Y"
- Use precise medical terminology
- Minimum 3 paragraphs"""

    user_prompt = f"""CONTEXT DOCUMENTS:
{context}

CLINICAL QUESTION: {query}

Provide a confident, structured, evidence-based clinical answer."""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        temperature=0.15,
        max_tokens=1024
    )

    answer = response.choices[0].message.content
    # Strip chain-of-thought reasoning tags (Qwen3)
    answer = re.sub(r"<think>.*?</think>", "", answer, flags=re.DOTALL).strip()
    return answer

# Quick test
_r = medical_rag("What are the effects of metformin on type 2 diabetes?", retriever="pinecone")
print(_r["answer"])

**Summary**  
Metformin lowers the risk of developing type 2 diabetes in high‑risk individuals, delays progression from prediabetes to overt diabetes, improves insulin sensitivity, and is an effective component of combination therapy for patients with established type 2 diabetes.

**Evidence**  
- The Diabetes Prevention Program (DPP) demonstrated that metformin “lowers the risk of type 2 diabetes in people with prediabetes, especially those who are younger and heavier and women who have had gestational diabetes” and delayed onset by about **2 years** (Doc 4).  
- In the same DPP trial, participants receiving metformin “also benefited” compared with placebo, showing a meaningful reduction in diabetes incidence, though the lifestyle‑change arm had the greatest effect (Doc 2).  
- Metformin “makes insulin work better” by enhancing peripheral insulin sensitivity, which is why it is used together with sulfonylureas and basal insulin (NPH) in uncontrolled type 2 diabetes, providing a “simpl

## Full RAG Pipeline

In [59]:
def medical_rag(query: str, use_reranking: bool = True, retriever: str = "pinecone"):
    # Step 1: Retrieve
    if retriever == "pinecone":
        results = retrieve_pinecone(query, top_k=10)
    elif retriever == "faiss":
        results = retrieve_faiss(query, top_k=10)
    elif retriever == "chroma":
        results = retrieve_chroma(query, top_k=10)
    else:
        raise ValueError(f"Unknown retriever: {retriever}")

    # Step 2: Re-rank or truncate
    results = rerank(query, results, top_k=5) if use_reranking else results[:5]

    # Step 3: Generate
    answer = generate_answer(query, results)

    return {
        "query":          query,
        "answer":         answer,
        "retrieved_docs": results,
        "retriever":      retriever,
        "reranking":      use_reranking,
    }

# Smoke test
_r = medical_rag("What are the effects of metformin on type 2 diabetes?", retriever="faiss")
print("FAISS pipeline OK ✓")
print(_r["answer"][:200])


FAISS pipeline OK ✓
**Summary**  
Metformin reduces the risk of progressing from pre‑diabetes to type 2 diabetes, delays disease onset by about 2 years, and improves insulin effectiveness; its benefit is most pronounced 


## RAGAS Evaluation

## alternative approach

In [60]:
from sentence_transformers import util

# --- helper: pick better GT doc by keyword + embedding ---
def find_gt_for_query(query, docs, max_candidates=5000):
    q_emb = embedder.encode(query, normalize_embeddings=True)
    
    # Use ALL docs, not just keyword-filtered subset
    # Keyword filtering is too aggressive and misses semantic matches
    candidate_docs = docs[:max_candidates]
    
    cand_embs = embedder.encode(
        [d["question"] for d in candidate_docs],
        normalize_embeddings=True,
        show_progress_bar=False,
        batch_size=512
    )
    sims = util.cos_sim(q_emb, cand_embs)[0]
    
    # Take top-3 and pick the one whose ANSWER is also most relevant
    top_indices = sims.topk(3).indices.tolist()
    best_idx = max(
        top_indices,
        key=lambda i: float(util.cos_sim(
            embedder.encode(query, normalize_embeddings=True),
            embedder.encode(candidate_docs[i]["answer"] or "", normalize_embeddings=True)
        ))
    )
    return candidate_docs[best_idx]



eval_queries = [
    "What are the effects of metformin on type 2 diabetes?",
    "How does aspirin affect cardiovascular disease?",
    "What is the role of vitamin D in bone health?",
    "Does exercise reduce depression symptoms?",
    "What are the side effects of statins?"
]

print("Running RAG pipeline on eval queries...")
results_list = [medical_rag(q) for q in eval_queries]

print("\n=== RAG Evaluation Results (with Ground Truth) ===\n")
total_relevancy = []
total_coverage = []
total_gt_similarity = []

for res in results_list:
    query = res["query"]

    # -------- better pseudo ground truth --------
    gt_doc = find_gt_for_query(query, docs)
    ground_truth = gt_doc.get("answer") or ""
    matched_question = gt_doc.get("question") or ""

    # safe texts
    answer_text = res.get("answer") or ""
    ctx_texts = [(d.get("text") or "") for d in res.get("retrieved_docs", [])]

    # embeddings (always lists)
    query_emb  = embedder.encode([query], normalize_embeddings=True)
    answer_emb = embedder.encode([answer_text], normalize_embeddings=True)
    gt_emb     = embedder.encode([ground_truth], normalize_embeddings=True)

    if not ctx_texts:
        print(f"Q: {query[:60]}... (no contexts, skipping metrics)\n")
        continue

    context_embs = embedder.encode(
        ctx_texts,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    # metrics
    relevancy = float(util.cos_sim(answer_emb, query_emb)[0][0])
    coverage  = float(util.cos_sim(answer_emb, context_embs).max())
    gt_sim    = float(util.cos_sim(answer_emb, gt_emb)[0][0])

    total_relevancy.append(relevancy)
    total_coverage.append(coverage)
    total_gt_similarity.append(gt_sim)

    print(f"Q:  {query[:60]}...")
    print(f"    Matched GT Question   : {matched_question[:80]}...")
    print(f"    Answer Relevancy      : {relevancy:.3f}")
    print(f"    Context Coverage      : {coverage:.3f}")
    print(f"    Ground Truth Similarity: {gt_sim:.3f}")
    print()

if total_relevancy:
    print("=" * 55)
    print(f"AVG Answer Relevancy       : {sum(total_relevancy)/len(total_relevancy):.3f}")
    print(f"AVG Context Coverage       : {sum(total_coverage)/len(total_coverage):.3f}")
    print(f"AVG Ground Truth Similarity: {sum(total_gt_similarity)/len(total_gt_similarity):.3f}")
else:
    print("No metrics computed (no valid retrieved contexts).")


Running RAG pipeline on eval queries...

=== RAG Evaluation Results (with Ground Truth) ===

Q:  What are the effects of metformin on type 2 diabetes?...
    Matched GT Question   : What are the genetic changes related to type 1 diabetes ?...
    Answer Relevancy      : 0.831
    Context Coverage      : 0.885
    Ground Truth Similarity: 0.677

Q:  How does aspirin affect cardiovascular disease?...
    Matched GT Question   : What is (are) warfarin sensitivity ?...
    Answer Relevancy      : 0.829
    Context Coverage      : 0.786
    Ground Truth Similarity: 0.755

Q:  What is the role of vitamin D in bone health?...
    Matched GT Question   : What are the genetic changes related to vitamin D-dependent rickets ?...
    Answer Relevancy      : 0.869
    Context Coverage      : 0.860
    Ground Truth Similarity: 0.838

Q:  Does exercise reduce depression symptoms?...
    Matched GT Question   : What are the treatments for ADCY5-related dyskinesia ?...
    Answer Relevancy      : 0.859

##  FastAPI (runs inside Kaggle)

In [61]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
import threading

nest_asyncio.apply()

app = FastAPI(title="Medical RAG API")

class QueryRequest(BaseModel):
    query: str
    retriever: str = "pinecone"
    use_reranking: bool = True

@app.get("/")
def root():
    return {"status": "Medical RAG API running ✅"}

@app.post("/ask")
def ask(req: QueryRequest):
    result = medical_rag(req.query, req.use_reranking, req.retriever)
    return {
        "query": result["query"],
        "answer": result["answer"],
        "top_docs": result["retrieved_docs"][:3],
        "retriever": result["retriever"],
        "reranking": result["reranking"]
    }

@app.get("/compare")
def compare(query: str):
    pinecone_results = retrieve_pinecone(query, top_k=5)
    faiss_results    = retrieve_faiss(query, top_k=5)
    chroma_results   = retrieve_chroma(query, top_k=5)
    return {
        "query":        query,
        "pinecone_top3": pinecone_results[:3],
        "faiss_top3":    faiss_results[:3],
        "chroma_top3":   chroma_results[:3],
    }


def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print("API running at http://localhost:8000 ✅")
print("Docs at http://localhost:8000/docs ✅")


API running at http://localhost:8000 ✅
Docs at http://localhost:8000/docs ✅


INFO:     Started server process [55]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use


In [51]:
# import requests

# # Test root
# r = requests.get("http://localhost:8000/")
# print("Root:", r.json())

# # Test /ask
# r2 = requests.post("http://localhost:8000/ask", json={
#     "query": "What are the effects of metformin on type 2 diabetes?",
#     "retriever": "pinecone",
#     "use_reranking": True
# })
# print("\n/ask response:")
# print(r2.json())

# # Test /compare
# r3 = requests.get("http://localhost:8000/compare", params={
#     "query": "does aspirin help heart disease?"
# })
# print("\n/compare response:")
# print(r3.json())


ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [62]:
from sentence_transformers import util

eval_queries = [
    "What are the effects of metformin on type 2 diabetes?",
    "How does aspirin affect cardiovascular disease?",
    "What is the role of vitamin D in bone health?",
    "Does exercise reduce depression symptoms?",
    "What are the side effects of statins?"
]

results_list = [medical_rag(q) for q in eval_queries]

print("\n=== RAG Evaluation Results ===")
total_relevancy = []
total_coverage = []

for res in results_list:
    answer_emb = embedder.encode(res['answer'], normalize_embeddings=True)
    query_emb = embedder.encode(res['query'], normalize_embeddings=True)
    context_embs = embedder.encode(
        [d['text'] for d in res['retrieved_docs']],
        normalize_embeddings=True
    )
    relevancy = float(util.cos_sim(answer_emb, query_emb))
    coverage = float(max(util.cos_sim(answer_emb, ctx) for ctx in context_embs))
    total_relevancy.append(relevancy)
    total_coverage.append(coverage)

    print(f"Q: {res['query'][:55]}...")
    print(f"   Answer Relevancy : {relevancy:.3f}")
    print(f"   Context Coverage : {coverage:.3f}")
    print()

print(f"AVG Answer Relevancy : {sum(total_relevancy)/len(total_relevancy):.3f}")
print(f"AVG Context Coverage : {sum(total_coverage)/len(total_coverage):.3f}")


INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


KeyboardInterrupt: 

In [53]:
!pip install -q gradio


In [65]:
import gradio as gr

def ask_interface(query, retriever, use_reranking):
    if not query.strip():
        return "Please enter a question.", ""
    try:
        result = medical_rag(query, use_reranking=use_reranking, retriever=retriever)
    except Exception as e:
        return f"**Error:** {str(e)}", ""

    formatted_answer = f"""{result['answer']}

---
**Retriever:** {retriever.capitalize()} &nbsp;|&nbsp; 
**Re-ranking:** {"Enabled" if use_reranking else "Disabled"} &nbsp;|&nbsp; 
**Model:** Llama 3.3 70B"""

    docs_text = ""
    for i, d in enumerate(result["retrieved_docs"][:3]):
        docs_text += f"**Doc {i+1}** &nbsp;&nbsp; Score: `{d['score']:.3f}`\n"
        docs_text += f"{d['text'][:300]}\n\n---\n"

    return formatted_answer, docs_text


with gr.Blocks(
    title="Medical RAG System",
    theme=gr.themes.Soft(
        primary_hue="purple",
        neutral_hue="slate",
    )
) as demo:
    gr.Markdown("# 🏥 Medical RAG System")
    gr.Markdown("Semantic search over MedQuAD · BGE Embeddings · Pinecone · FAISS · ChromaDB · Llama 3.3 70B")

    with gr.Row():
        with gr.Column(scale=2):
            query_box = gr.Textbox(
                label="Medical Question",
                placeholder="e.g. What are the effects of metformin on type 2 diabetes?",
                lines=3
            )
        with gr.Row():
            retriever = gr.Radio(
                ["pinecone", "faiss", "chroma"],   # ← chroma added
                label="Vector Store",
                value="pinecone"
            )
            reranking = gr.Checkbox(label="⚡ Cross-encoder Re-ranking", value=True)

    submit_btn = gr.Button("🔍 Ask", variant="primary", size="lg")

    with gr.Row():
        with gr.Column():
            answer_box = gr.Markdown(label="Generated Answer", value="Your answer will appear here...")
        with gr.Column():
            docs_box = gr.Markdown(label="Retrieved Documents", value="Retrieved documents will appear here...")

    gr.Markdown("### 💡 Example Questions")
    gr.Examples(
        examples=[
            ["What are the effects of metformin on type 2 diabetes?",  "pinecone", True],
            ["How does aspirin affect cardiovascular disease?",          "pinecone", True],
            ["What is the role of vitamin D in bone health?",            "faiss",    True],
            ["Does exercise reduce depression symptoms?",                 "chroma",   True],
            ["What are the side effects of statins?",                    "faiss",    False],
        ],
        inputs=[query_box, retriever, reranking],
    )

    submit_btn.click(
        fn=ask_interface,
        inputs=[query_box, retriever, reranking],
        outputs=[answer_box, docs_box],
    )

demo.launch(share=True)


/tmp/ipykernel_55/3961629214.py:26: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7865
* Running on public URL: https://4d3590f6a58ea81834.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
